# Transactions — 03: Soft Delete by external_id

**Persona:** Data Pipeline Engineer

**Goal:** Demonstrate the full lifecycle of an item that needs to be retracted:

1. Ingest an item carrying an `external_id` and an associated asset
2. Resolve the item's internal ID using a CQL2 filter on `external_id`
3. Soft-delete the item — it becomes invisible to queries but its row is retained
4. Confirm the associated asset record **survives** the item soft-delete
5. Demonstrate the asset blocking-reference guard before final cleanup

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Catalog `CATALOG_ID` and collection `COLLECTION_ID` exist with a LayerConfig
- `DYNASTORE_WRITE_TOKEN` set if the instance requires authentication
- No existing item with `external_id = 'RFE2_ERR_2024-01-01'` in the target collection

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_WRITE_TOKEN` | _(empty)_ | Bearer token for write/delete operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `rfe2-rainfall` | Target collection |

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
WRITE_TOKEN   = os.environ.get("DYNASTORE_WRITE_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "rfe2-rainfall")

EXTERNAL_ID = "RFE2_ERR_2024-01-01"

headers = {"Authorization": f"Bearer {WRITE_TOKEN}"} if WRITE_TOKEN else {}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30)

print(f"Base URL     : {BASE_URL}")
print(f"Catalog      : {CATALOG_ID}")
print(f"Collection   : {COLLECTION_ID}")
print(f"external_id  : {EXTERNAL_ID}")
print(f"Auth header  : {'set' if WRITE_TOKEN else 'not set'}")

## Step 1 — Ingest a test item with external_id

Create a minimal RFE2 rainfall estimate item. The `external_id` property is the
stable identity key used by DynaStore's write-policy matcher. It is what allows
a pipeline to re-ingest the same logical record idempotently (upsert) and to
resolve the internal database ID from a human-readable business key.

In [ ]:
item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": EXTERNAL_ID,
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [33.0, 3.0],
                [42.0, 3.0],
                [42.0, 12.0],
                [33.0, 12.0],
                [33.0, 3.0]
            ]
        ]
    },
    "bbox": [33.0, 3.0, 42.0, 12.0],
    "properties": {
        "datetime": "2024-01-01T00:00:00Z",
        "platform": "rfe2",
        "product": "rainfall_estimate",
        "external_id": EXTERNAL_ID,
        "status": "erroneous"
    },
    "assets": {},
    "links": []
}

resp = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=item,
)
assert resp.status_code == 201, (
    f"Expected 201 Created, got {resp.status_code}: {resp.text[:400]}"
)

created_item = resp.json()
item_id = created_item.get("id", EXTERNAL_ID)
print(f"Item ingested — id: {item_id}")

In [ ]:
# Register an associated asset for this item.
# Assets survive item soft-delete; this is the key behaviour demonstrated later.
asset_body = {
    "href": f"gs://fao-rfe2-data/2024/01/01/{EXTERNAL_ID}.tif",
    "type": "image/tiff; application=geotiff",
    "roles": ["data"],
    "title": "RFE2 Rainfall Estimate GeoTIFF",
    "item_id": item_id,
    "extra_fields": {
        "eo:bands": [{"name": "B1", "description": "rainfall_mm"}]
    }
}

resp_asset = client.post(
    f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    json=asset_body,
)
assert resp_asset.status_code == 201, (
    f"Expected 201 Created for asset, got {resp_asset.status_code}: {resp_asset.text[:400]}"
)

created_asset = resp_asset.json()
asset_id = created_asset.get("id") or created_asset.get("asset_id")
print(f"Asset registered — id: {asset_id}")
print(json.dumps(created_asset, indent=2))

## Step 2 — Resolve item by external_id via CQL2

When a pipeline knows only the business key (`external_id`) but not the internal
DynaStore item ID, use a CQL2-Text filter to locate it. The filter runs through
DynaStore's pygeofilter pipeline and translates to a parameterised SQL predicate.

Filter syntax: `external_id='<value>'` with `filter-lang=cql2-text`.

In [ ]:
resp = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={
        "filter": f"external_id='{EXTERNAL_ID}'",
        "filter-lang": "cql2-text",
        "limit": 5,
    },
)
assert resp.status_code == 200, (
    f"Expected 200 OK, got {resp.status_code}: {resp.text[:400]}"
)

result = resp.json()
matched_features = result.get("features", [])
assert len(matched_features) >= 1, (
    f"CQL2 filter returned 0 features — item with external_id={EXTERNAL_ID!r} not found"
)

resolved_item_id = matched_features[0]["id"]
print(f"Resolved external_id={EXTERNAL_ID!r} → item_id={resolved_item_id!r}")
assert resolved_item_id == item_id, (
    f"Resolved ID {resolved_item_id!r} does not match ingested ID {item_id!r}"
)

## Step 3 — Soft delete the item

A DELETE on the item endpoint performs a **soft delete**: the row in PostgreSQL
is marked as deleted (e.g. `deleted_at` timestamp set) but is not physically
removed. The item becomes invisible to all standard query endpoints immediately.

Soft-deleted rows persist until a manual purge is executed (see Notes cell below).

In [ ]:
resp = client.delete(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_id}"
)
assert resp.status_code == 204, (
    f"Expected 204 No Content, got {resp.status_code}: {resp.text[:400]}"
)
print(f"Soft-deleted item {item_id!r} — status {resp.status_code}")

In [ ]:
# Confirm item is no longer visible via the standard GET endpoint
resp = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_id}"
)
assert resp.status_code == 404, (
    f"Expected 404 Not Found after soft-delete, got {resp.status_code}: {resp.text[:400]}"
)
print(f"GET {item_id!r} after soft-delete → 404 Not Found (correct)")

# Also confirm it disappears from the CQL2 filter results
resp_filter = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={
        "filter": f"external_id='{EXTERNAL_ID}'",
        "filter-lang": "cql2-text",
    },
)
assert resp_filter.status_code == 200
filter_hits = resp_filter.json().get("features", [])
assert len(filter_hits) == 0, (
    f"Expected 0 results after soft-delete, got {len(filter_hits)}"
)
print(f"CQL2 filter on external_id after soft-delete → 0 results (correct)")

## Step 4 — Verify asset is NOT deleted

Assets have an independent lifecycle from items. A soft-delete on the item does **not**
cascade to associated assets. This preserves the storage reference and audit trail —
the physical file in GCS/local storage is unaffected and the asset record remains
queryable via the Assets API.

In [ ]:
resp = client.get(
    f"/assets/catalogs/{CATALOG_ID}/assets/{asset_id}"
)
assert resp.status_code == 200, (
    f"Expected 200 OK — asset should survive item soft-delete. "
    f"Got {resp.status_code}: {resp.text[:400]}"
)

surviving_asset = resp.json()
print(f"Asset {asset_id!r} still accessible after item soft-delete (correct)")
print(f"  href  : {surviving_asset.get('href')}")
print(f"  type  : {surviving_asset.get('type')}")
print(f"  roles : {surviving_asset.get('roles')}")

## Edge cases

### Case A — DELETE asset with cascade_delete=False (blocking reference)

Attempting to delete an asset that is still referenced by a collection record
without allowing cascade should be rejected with `409 Conflict`.
This guard prevents orphaned collection rows pointing to non-existent assets.

In [ ]:
resp = client.delete(
    f"/assets/catalogs/{CATALOG_ID}/assets/{asset_id}",
    params={"cascade_delete": "false"},
)
# Expected: 409 Conflict — the asset has a blocking reference.
# On platforms where the asset is already unlinked from the soft-deleted item
# this may return 204; document the actual behaviour.
print(f"DELETE asset with cascade_delete=false → {resp.status_code}")
if resp.status_code == 409:
    print("  409 Conflict — blocking reference guard active (expected)")
    print(f"  Detail: {resp.text[:300]}")
elif resp.status_code == 204:
    print(
        "  204 No Content — platform unlinked asset on item soft-delete; "
        "no blocking reference remains."
    )
    # Asset was already deleted; mark asset_id as None so teardown skips it
    asset_id = None
else:
    print(f"  Unexpected status {resp.status_code}: {resp.text[:300]}")

## Notes — soft-delete retention

- Soft-deleted item rows persist in the database indefinitely.
- **No automatic purge scheduler exists** in the current platform.
- A future `VACUUM` or admin purge API call is required to physically remove them.
- The associated asset record and physical storage object (GCS/local) are **never**
  touched by item soft-delete; they must be cleaned up explicitly via the Assets API.
- To fully expunge an item and all associated data use:
  `DELETE /assets/catalogs/{catalog_id}/assets/{asset_id}?force=true&propagate=true`
  followed by a hard-delete of the item row (admin endpoint, not exposed in standard API).

## Teardown

Force-delete the asset with propagation to clean up all references and the storage record.

In [ ]:
if asset_id is not None:
    resp = client.delete(
        f"/assets/catalogs/{CATALOG_ID}/assets/{asset_id}",
        params={"force": "true", "propagate": "true"},
    )
    assert resp.status_code == 204, (
        f"Expected 204 No Content on force-delete, got {resp.status_code}: {resp.text[:400]}"
    )
    print(f"Asset {asset_id!r} force-deleted with propagation — status {resp.status_code}")

    # Confirm asset is gone
    resp_check = client.get(
        f"/assets/catalogs/{CATALOG_ID}/assets/{asset_id}"
    )
    assert resp_check.status_code == 404, (
        f"Expected 404 after force-delete, got {resp_check.status_code}"
    )
    print(f"Confirmed: GET asset {asset_id!r} → 404 Not Found")
else:
    print("Asset was already removed in edge-case cell — no teardown needed")

client.close()